# ReverseEngineer Coursera GraphQL and upload to GCS and BigQuery

In [ ]:
# Install required packages if needed (uncomment to run)
# !pip install -q requests pandas google-cloud-storage google-cloud-bigquery
import os
import json
import requests
import pandas as pd

In [ ]:
# GraphQL request to Coursera inferred gateway (Search operation)
endpoint = 'https://www.coursera.org/graphql-gateway?opname=Search'
search_query = 'python'  # change as needed
limit = 50
GRAPHQL_QUERY = '''query Search($requests: [Search_Request!]!) {
  SearchResult {
    search(requests: $requests) {
      elements {
        id
        name
        url
        imageUrl
        productType
        productDifficultyLevel
        productDuration
        avgProductRating
        numProductRatings
        skills
        partners
        isPartOfCourseraPlus
        isCourseFree
        isCreditEligible
        translatedName
        tagline
      }
    }
  }
}
'''
payload = {
    'operationName': 'Search',
    'variables': {'requests': [{'query': search_query, 'limit': limit}]},
    'query': GRAPHQL_QUERY,
}
headers = {'Content-Type': 'application/json'}
resp = requests.post(endpoint, json=payload, headers=headers, timeout=30)
resp.raise_for_status()
data = resp.json()
# Inspect top-level keys
print('Top-level keys in response:', list(data.keys()))
# Extract search results robustly
search_results = []
d = data.get('data', {})
sr = d.get('SearchResult', {})
if sr:
    search_list = sr.get('search') or []
    for block in search_list:
        elems = block.get('elements') or []
        for e in elems:
            search_results.append(e)
len(search_results)

In [ ]:
# Convert the results to a DataFrame and normalize nested fields where useful
def safe_get(d, k):
    return d.get(k) if isinstance(d, dict) else None
rows = []
for e in search_results:
    rows.append({
        'id': safe_get(e, 'id'),
        'name': safe_get(e, 'name'),
        'url': safe_get(e, 'url'),
        'imageUrl': safe_get(e, 'imageUrl'),
        'productType': safe_get(e, 'productType'),
        'difficulty': safe_get(e, 'productDifficultyLevel'),
        'duration': safe_get(e, 'productDuration'),
        'avgRating': safe_get(e, 'avgProductRating'),
        'numRatings': safe_get(e, 'numProductRatings'),
        'skills': ','.join(e.get('skills') or []) if isinstance(e.get('skills'), list) else e.get('skills'),
        'partners': ','.join(e.get('partners') or []) if isinstance(e.get('partners'), list) else e.get('partners'),
        'isPartOfCourseraPlus': safe_get(e, 'isPartOfCourseraPlus'),
        'isCourseFree': safe_get(e, 'isCourseFree'),
        'isCreditEligible': safe_get(e, 'isCreditEligible'),
        'tagline': safe_get(e, 'tagline'),
        'translatedName': safe_get(e, 'translatedName'),
    })
df = pd.DataFrame(rows)
print('Rows fetched:', len(df))
df.head()

In [ ]:
# Save DataFrame to CSV locally
csv_path = 'courses.csv'
df.to_csv(csv_path, index=False)
print('Saved CSV to', csv_path)

## Upload CSV to GCS

The following cell uploads the CSV to a GCS bucket. Ensure `GOOGLE_APPLICATION_CREDENTIALS` points to a service-account JSON with `storage.objects.create` permission, or your environment is already authenticated. Set `bucket_name` accordingly.

In [ ]:
from google.cloud import storage
bucket_name = os.environ.get('GCS_BUCKET') or 'my-bucket-name'  # change to your bucket
destination_blob_name = 'coursera_exports/' + os.path.basename(csv_path)
if bucket_name == 'my-bucket-name':
    print('Set environment variable GCS_BUCKET or change bucket_name in the cell before running')
else:
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(csv_path)
    gcs_uri = f'gs://{bucket_name}/{destination_blob_name}'
    print('Uploaded to', gcs_uri)

## Load CSV into BigQuery

The next cell loads the CSV from GCS into BigQuery. Ensure your service account has BigQuery permissions. Set `BQ_DATASET` and `BQ_TABLE` or change the variables in the cell.

In [ ]:
from google.cloud import bigquery
bq_client = bigquery.Client()
project = bq_client.project
dataset_id = os.environ.get('BQ_DATASET') or 'my_dataset'  # change
table_id = os.environ.get('BQ_TABLE') or 'coursera_courses'  # change
full_table_id = f'{project}.{dataset_id}.{table_id}'
if 'my_dataset' in dataset_id or 'coursera_courses' in table_id:
    print('Set BQ_DATASET and BQ_TABLE environment vars or edit the cell to target your dataset/table')
else:
    gcs_uri = f'gs://{bucket_name}/{destination_blob_name}'
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        autodetect=True,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )
    load_job = bq_client.load_table_from_uri(gcs_uri, full_table_id, job_config=job_config)
    print('Starting BigQuery load job...')
    load_job.result()
    table = bq_client.get_table(full_table_id)
    print('Loaded', table.num_rows, 'rows into', full_table_id)